# Mini Pupper Text-to-Speech (gTTS)

Use the textbox and click **Speak** to generate speech with gTTS.
This notebook tries to play audio on the machine speaker first (for the robot).
If local playback tools are unavailable, it falls back to an in-notebook audio player.

In [ ]:
# Run once if needed:
# %pip install gTTS ipywidgets

import os
import shutil
import subprocess
import tempfile
from pathlib import Path

from gtts import gTTS
import ipywidgets as widgets
from IPython.display import Audio, display

In [ ]:
# Optional defaults
LANG = "en"
SLOW = False
SAVE_DIR = Path.cwd() / "tts_output"
SAVE_DIR.mkdir(exist_ok=True)

def play_local_audio(audio_path: str) -> bool:
    """Try local speaker playback; return True if successful."""
    players = [
        ["mpg123", audio_path],
        ["ffplay", "-nodisp", "-autoexit", "-loglevel", "quiet", audio_path],
        ["play", "-q", audio_path],
        ["aplay", audio_path],
    ]

    for cmd in players:
        if shutil.which(cmd[0]):
            try:
                subprocess.run(cmd, check=True)
                return True
            except Exception:
                continue
    return False

In [ ]:
%%bash

amixer -c 0 sset 'PCM' 100%

In [ ]:
text_box = widgets.Textarea(
    value="Hello! I am your robot dog.",
    placeholder="Type text for the robot to speak...",
    description="Text:",
    layout=widgets.Layout(width="100%", height="120px")
)

lang_box = widgets.Text(
    value=LANG,
    description="Lang:",
    tooltip="gTTS language code (e.g., en, es, fr, ja)"
)

speak_button = widgets.Button(
    description="Speak",
    button_style="success",
    icon="volume-up"
)

output = widgets.Output()

def on_speak_click(_):
    with output:
        output.clear_output()

        text = text_box.value.strip()
        lang = lang_box.value.strip() or "en"

        if not text:
            print("Please enter some text first.")
            return

        # Create unique output file each click
        fd, tmp_path = tempfile.mkstemp(suffix=".mp3", dir=str(SAVE_DIR))
        os.close(fd)

        try:
            tts = gTTS(text=text, lang=lang, slow=SLOW)
            tts.save(tmp_path)
            print(f"Saved speech to: {tmp_path}")

            played = play_local_audio(tmp_path)
            if played:
                print("Played on local speaker.")
            else:
                print("No local audio player found (or playback failed). Using notebook audio player:")
                display(Audio(filename=tmp_path, autoplay=True))

        except Exception as e:
            print(f"TTS failed: {e}")

speak_button.on_click(on_speak_click)

display(widgets.VBox([text_box, lang_box, speak_button, output]))

## Notes

- gTTS requires internet access to generate speech.
- For robot-local playback, install one of: `mpg123`, `ffplay`, `sox` (`play`), or use existing audio tooling on your image.
- If you want this to send text over network to another machine for playback, we can add a UDP/HTTP sender cell next.